# Tips 06：Excel帳票を自動作成

**業務の困りごと**：毎月同じ形式の売上報告書を手作業で作る。

**このTipsでできること**：`openpyxl` はセル単位で値や書式を指定できます。見出しの色、列幅、金額の表示形式と合計数式を設定します。

**使用ライブラリ**：openpyxl

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

商品名と金額の組を3件用意します。これは帳票に転記する前の元データです。

最後の `head()` で先頭の行を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
from openpyxl import Workbook,load_workbook
from openpyxl.styles import Font,PatternFill
from openpyxl.utils import get_column_letter
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 帳票に載せる商品と金額を用意する
rows=[('茶',1200),('菓子',900),('水',500)]
import pandas as pd
print('帳票に使う元データ（先頭5行）:')
pd.DataFrame(rows,columns=['商品','売上金額']).head()


## 2. 自動処理

`openpyxl` はセル単位で値や書式を指定できます。見出しの色、列幅、金額の表示形式と合計数式を設定します。

**結果を見るポイント**：B5セルには `=SUM(B2:B4)` が保存されます。数式の計算結果はExcelで開いて確認してください。


In [ ]:
# Excelブックとシートを新しく作る
wb = Workbook()
ws = wb.active
ws.title = '月次売上'
ws.append(['商品', '売上金額'])
for row in rows:
    ws.append(row)
# 合計欄にはSUM数式を入れる
ws['A5']='合計'; ws['B5']='=SUM(B2:B4)'
# 見出し行の文字色と背景色を設定する
for cell in ws[1]:
    cell.font=Font(bold=True,color='FFFFFF')
    cell.fill=PatternFill('solid',fgColor='23549B')
ws.column_dimensions['A'].width=18; ws.column_dimensions['B'].width=18
for r in range(2,6): ws[f'B{r}'].number_format='#,##0" 円"'
# 帳票をExcelファイルとして保存する
out=base/'Tips06_月次帳票.xlsx'; wb.save(out)
print('作成:',out)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

B5セルには `=SUM(B2:B4)` が保存されます。数式の計算結果はExcelで開いて確認してください。


In [ ]:
# 件数や合計が期待値と一致するか検査する
check=load_workbook(out)
assert check.active['B5'].value=='=SUM(B2:B4)'
print('確認OK。Excelで開くと数式が計算されます。')


## 4. 持ち帰り・練習

**練習問題**：見出しに報告月を追加しましょう。

openpyxlは数式を保存しますが、Python上では数式を計算しません。

**実務で使う前に**：`openpyxl` 自体はExcelの数式を計算しません。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
